# Significance Testing

Stage 4 artifact reader. The reusable implementations live in `src/significance.py`; saved tables are generated by `scripts/run_significance.py`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import config
from scripts.run_significance import generate_significance_artifacts

results_dir = Path(config.DATA_RESULTS_DIR)
features_dir = Path(config.DATA_FEATURES_DIR)

## Refresh Artifacts

In [2]:
paths = generate_significance_artifacts(
    results_dir=results_dir,
    features_dir=features_dir,
    block_size=8,
    n_bootstrap=5000,
    seed=42,
)
pd.DataFrame({'artifact': paths.keys(), 'path': [str(p) for p in paths.values()]})

,artifact,path
0,weekly_errors,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
1,dm_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
2,bootstrap_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
3,summary,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...


## Weekly Model Errors

In [3]:
weekly_errors = pd.read_parquet(results_dir / 'weekly_model_errors.parquet')
weekly_error_summary = (
    weekly_errors.groupby('model')
    .agg(mean_weekly_mse=('weekly_mse', 'mean'), n_weeks=('week', 'nunique'), min_stocks=('n_stocks', 'min'))
    .sort_values('mean_weekly_mse')
)
display(weekly_error_summary)
display(weekly_errors.head(12))

,mean_weekly_mse,n_weeks,min_stocks
model,,,
GNN-Ensemble,0.032015,103,465
GNN-Correlation,0.032191,103,465
LSTM,0.032424,103,465
HAR per-stock,0.032858,103,465
HAR pooled,0.033104,103,465
GNN-Sector,0.033631,103,465
GNN-Granger,0.033702,103,465
Rank-loss GNN-Granger,0.138732,103,465
Rank-loss GNN-Sector,0.182689,103,465


,week,model,experiment_id,model_family,graph_type,loss_type,feature_version,graph_version,prediction_path,weekly_mse,n_stocks
0,2024-01-01,GNN-Correlation,baseline_gnn_correlation,GNN,correlation,mse,stock_features_v1,correlation_threshold_0.3_lookback_252,data/results/test_preds_gnn_corr.parquet,0.017233,465
1,2024-01-01,GNN-Ensemble,baseline_gnn_ensemble,GNN ensemble,correlation+sector+granger,mse,stock_features_v1,correlation+sector+granger_v1,data/results/test_preds_gnn_ensemble.parquet,0.018573,465
2,2024-01-01,GNN-Granger,baseline_gnn_granger,GNN,granger,mse,stock_features_v1,granger_lag_5_bonferroni,data/results/test_preds_gnn_granger.parquet,0.020228,465
3,2024-01-01,GNN-Sector,baseline_gnn_sector,GNN,sector,mse,stock_features_v1,sector_canonical_gics_labels_v1,data/results/test_preds_gnn_sector.parquet,0.021086,465
4,2024-01-01,HAR per-stock,baseline_har_per_stock,HAR,none,squared_error,har_realized_volatility_lags_v1,none,data/results/test_preds_har.parquet,0.016564,465
5,2024-01-01,HAR pooled,baseline_har_pooled,HAR,none,squared_error,har_realized_volatility_lags_v1,none,data/results/test_preds_har_pooled.parquet,0.016164,465
6,2024-01-01,LSTM,baseline_lstm,LSTM,none,mse,stock_features_v1,none,data/results/test_preds_lstm.parquet,0.016731,465
7,2024-01-01,Rank-loss GNN-Correlation,rankloss_gnn_correlation,GNN,correlation,bpr_rank,stock_features_v1,correlation_threshold_0.3_lookback_252,data/results/test_preds_gnn_corr_rankloss.parquet,0.221394,465
8,2024-01-01,Rank-loss GNN-Granger,rankloss_gnn_granger,GNN,granger,bpr_rank,stock_features_v1,granger_lag_5_bonferroni,data/results/test_preds_gnn_granger_rankloss.p...,0.119816,465
9,2024-01-01,Rank-loss GNN-Sector,rankloss_gnn_sector,GNN,sector,bpr_rank,stock_features_v1,sector_canonical_gics_labels_v1,data/results/test_preds_gnn_sector_rankloss.pa...,0.172711,465


## Diebold-Mariano Tests

In [4]:
dm_results = pd.read_csv(results_dir / 'dm_test_results.csv')
dm_view = dm_results.sort_values(['baseline', 'p_value_bh', 'p_value'])
display(dm_view)
display(dm_view[dm_view['rejected_bh']])

,model,baseline,dm_stat,p_value,n_weeks,mean_loss_diff,p_value_bh,rejected_bh
0,GNN-Ensemble,HAR per-stock,0.713396,0.238615,103,0.000843,0.909359,False
1,GNN-Correlation,HAR per-stock,0.646150,0.259817,103,0.000667,0.909359,False
2,GNN-Sector,HAR per-stock,-0.568345,0.714475,103,-0.000773,1.000000,False
3,GNN-Granger,HAR per-stock,-0.650531,0.741594,103,-0.000844,1.000000,False
4,Rank-loss GNN-Correlation,HAR per-stock,-28.777604,1.000000,103,-0.188947,1.000000,False
5,Rank-loss GNN-Granger,HAR per-stock,-57.179248,1.000000,103,-0.105874,1.000000,False
6,Rank-loss GNN-Sector,HAR per-stock,-48.436956,1.000000,103,-0.149831,1.000000,False
7,GNN-Ensemble,HAR pooled,0.897046,0.185903,103,0.001089,0.682069,False
8,GNN-Correlation,HAR pooled,0.863746,0.194877,103,0.000914,0.682069,False
9,GNN-Sector,HAR pooled,-0.377009,0.646525,103,-0.000527,1.000000,False


,model,baseline,dm_stat,p_value,n_weeks,mean_loss_diff,p_value_bh,rejected_bh


## Bootstrap Sharpe Confidence Intervals

In [5]:
bootstrap_ci = pd.read_csv(results_dir / 'bootstrap_sharpe_ci.csv')
display(
    bootstrap_ci[bootstrap_ci['comparison'] == 'sharpe']
    .sort_values(['strategy', 'point_estimate'], ascending=[True, False])
)
display(
    bootstrap_ci[bootstrap_ci['comparison'] == 'sharpe_diff']
    .sort_values(['strategy', 'ci_lower'], ascending=[True, False])
)

,strategy,return_path,model,comparison,benchmark,point_estimate,ci_lower,ci_upper,n_weeks,block_size,n_bootstrap
0,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,Equal-weight,sharpe,NaN,0.878126,-0.201938,2.166350,103,8,5000
3,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Granger,sharpe,NaN,0.811128,-0.258544,2.080357,103,8,5000
2,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Ensemble,sharpe,NaN,0.806433,-0.249003,2.074990,103,8,5000
7,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM,sharpe,NaN,0.805721,-0.253608,2.075955,103,8,5000
4,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Sector,sharpe,NaN,0.798161,-0.255396,2.069568,103,8,5000
6,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,HAR pooled,sharpe,NaN,0.797788,-0.260945,2.068129,103,8,5000
5,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,HAR per-stock,sharpe,NaN,0.792135,-0.256616,2.052169,103,8,5000
1,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation,sharpe,NaN,0.778344,-0.271070,2.041903,103,8,5000
21,long_short,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM,sharpe,NaN,-0.948752,-2.416177,0.487822,103,8,5000
18,long_short,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Sector,sharpe,NaN,-0.973775,-2.488190,0.459018,103,8,5000


,strategy,return_path,model,comparison,benchmark,point_estimate,ci_lower,ci_upper,n_weeks,block_size,n_bootstrap
10,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Granger,sharpe_diff,Equal-weight,-0.066998,-0.189827,0.055078,103,8,5000
9,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Ensemble,sharpe_diff,Equal-weight,-0.071693,-0.206158,0.065036,103,8,5000
14,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM,sharpe_diff,Equal-weight,-0.072406,-0.206696,0.073783,103,8,5000
13,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,HAR pooled,sharpe_diff,Equal-weight,-0.080338,-0.222660,0.063997,103,8,5000
8,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation,sharpe_diff,Equal-weight,-0.099783,-0.239846,0.038033,103,8,5000
11,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Sector,sharpe_diff,Equal-weight,-0.079965,-0.250084,0.086940,103,8,5000
12,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,HAR per-stock,sharpe_diff,Equal-weight,-0.085992,-0.270769,0.103578,103,8,5000


## Significance Summary

In [6]:
summary = pd.read_csv(results_dir / 'significance_summary.csv')
display(summary)

,section,metric,value,details
0,dm_tests,fdr_significant_model_vs_baseline,0.000000,21 DM comparisons at FDR 0.05
1,dm_tests,min_bh_adjusted_p,0.682069,One-sided lower-loss alternative
2,bootstrap,positive_sharpe_diff_ci,0.000000,7 Sharpe-difference intervals vs available ben...
